# 00 Data Sanity

## Why This Notebook Exists
This is the first notebook to run in any workflow. It verifies that feature panels are populated, aligned, and numerically sane before you spend time on backtests.

## What You Should Confirm
- Each pair has non-empty feature rows.
- Date ranges are reasonable and overlap with your intended sample window.
- Required columns are present and free of NaN values.
- Leveraged leg daily return volatility is generally higher than spot (quick reality check).


In [ ]:
# Standard notebook bootstrap.
# Goal: make imports and file paths work even if notebook is opened from /notebooks.
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.experiments import build_processed_data, pair_list
from src.utils.io import load_config


## Configuration Controls
- Set `CONFIG_PATH` to `default.yaml`, `btc_bitx.yaml`, or `qqq_tqqq.yaml`.
- Keep `TRY_BUILD_IF_MISSING=True` to auto-run feature construction if CSVs are absent.
- If Yahoo rate-limits during build, re-run after a short wait.


In [ ]:
CONFIG_PATH = ROOT / "configs" / "default.yaml"
TRY_BUILD_IF_MISSING = True

cfg = load_config(CONFIG_PATH)
pairs = pair_list(cfg)
print("Using config:", CONFIG_PATH)
print("Pairs:", [p["name"] for p in pairs])


In [ ]:
def load_features_local_or_build(cfg: dict, try_build: bool = True) -> dict[str, pd.DataFrame]:
    """Load processed feature CSV files; optionally build them through the pipeline."""
    processed_dir = ROOT / "data" / "processed"
    paths = {p["name"]: processed_dir / f"{p['name']}_features.csv" for p in pair_list(cfg)}

    # Fast path: all expected feature files already exist.
    if all(path.exists() for path in paths.values()):
        out = {}
        for name, path in paths.items():
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            df.index = pd.to_datetime(df.index, utc=True)
            out[name] = df
        return out

    if not try_build:
        missing = [str(path) for path in paths.values() if not path.exists()]
        raise FileNotFoundError(f"Missing processed features: {missing}")

    # Slow path: build from raw data.
    # Note: this can fail if data source is rate-limited or unsupported for a ticker/date range.
    print("Processed features missing; building from pipeline...")
    return build_processed_data(cfg)


In [ ]:
features_by_pair = load_features_local_or_build(cfg, try_build=TRY_BUILD_IF_MISSING)

rows = []
for pair in pairs:
    name = pair["name"]
    df = features_by_pair[name]
    rows.append({
        "pair": name,
        "rows": len(df),
        "start": df.index.min(),
        "end": df.index.max(),
        "missing_total": int(df.isna().sum().sum()),
        "spot_price_min": float(df["spot_adj_close"].min()),
        "lev_price_min": float(df["lev_adj_close"].min()),
        "spot_ret_std": float(df["spot_ret"].std()),
        "lev_ret_std": float(df["lev_ret"].std()),
    })

summary = pd.DataFrame(rows).set_index("pair").sort_index()
summary


In [ ]:
# Column-level missingness helps pinpoint where feature creation may be dropping rows.
for pair in pairs:
    name = pair["name"]
    df = features_by_pair[name]
    print(f"\n{name}: top missing columns")
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame("missing_ratio"))


## Interpretation Tips
- If `rows` is unexpectedly low, inspect config dates and lookback windows (hedge + regimes).
- If `missing_total > 0`, inspect dropped columns before running strategy logic.
- If `lev_ret_std <= spot_ret_std` for long stretches, verify ticker mapping and adjusted close handling.


In [ ]:
# Visual checks: price trajectories and return distributions.
for pair in pairs:
    name = pair["name"]
    df = features_by_pair[name]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(df.index, df["spot_adj_close"], label="Spot", lw=1.5)
    axes[0].plot(df.index, df["lev_adj_close"], label="Leveraged", lw=1.2)
    axes[0].set_title(f"{name}: adjusted close")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    axes[1].hist(df["spot_ret"], bins=60, alpha=0.6, label="Spot")
    axes[1].hist(df["lev_ret"], bins=60, alpha=0.6, label="Leveraged")
    axes[1].set_title(f"{name}: daily return distribution")
    axes[1].legend()
    axes[1].grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


In [ ]:
# Required fields for downstream strategy/backtest notebooks.
required_cols = [
    "spot_adj_close",
    "lev_adj_close",
    "spot_ret",
    "lev_ret",
    "drift",
    "vol",
    "beta",
]

for name, df in features_by_pair.items():
    assert not df.empty, f"{name}: empty feature panel"
    assert set(required_cols).issubset(df.columns), f"{name}: missing required columns"
    assert (df["spot_adj_close"] > 0).all(), f"{name}: non-positive spot prices"
    assert (df["lev_adj_close"] > 0).all(), f"{name}: non-positive leveraged prices"
    assert df[required_cols].notna().all().all(), f"{name}: NaNs remain in required features"

print("Data sanity checks passed.")


## Next Notebook
If this notebook passes, continue to `01_strategy_baseline.ipynb`. If it fails, fix data coverage first; strategy outputs are not meaningful with incomplete features.
